# 純 3D 流形不連續伽遼金 (Manifold DG) 球面平流求解器
## 【JAX 極限加速與正交實驗版 - Simplex 三角形網格修復版】

本筆記本實現了球面平流方程的 3D 流形不連續伽遼金 (DG) 求解器，基於嚴格的非結構化三角形 (Simplex) 網格與 Dubiner 正交多項式基底。

### 🧪 實驗規劃與優化特點：
1. 6-Case 正交實驗矩陣：
   - Formulations (配方): divergence (守恆散度), split2 (雙分裂形式), split3 (滿足 SBP 的三分裂形式)
   - Flux Types (通量): upwind (迎風耗散通量), central (無耗散對稱通量)
2. 極長週期模擬：模擬時間設為 120 天 (約10次完整球面旋轉)，網格配置為 n_div_list = [4, 8, 16, 32, 64]。
3. 嚴謹數學核心：使用 5-Stage Runge-Kutta 確保時間與空間高精度匹配，並嚴格計算流形度規。
4. 極限 VRAM 控管：計算完畢後立即落盤為 .npz 並釋放記憶體。

In [1]:
# STREAMING_CHUNK:Configuring environment and JAX...
import sys
import os
import gc
import time
import numpy as np
import pandas as pd

# 確保套件安裝與路徑設定 (若尚未安裝則自動 clone)
if not os.path.exists('Simplex-DG-solver'):
    os.system("git clone https://github.com/wcw100168/Simplex-DG-solver.git")
    os.chdir("Simplex-DG-solver")
    os.system("pip install -e .")
else:
    sys.path.append(os.path.abspath('.'))
    os.chdir("Simplex-DG-solver")
    os.system("pip install -e .")

os.environ["JAX_PLATFORMS"] = "mps,cpu"

# 啟用 JAX 雙精度以滿足守恆誤差分析精度需求 (10^-16)
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import jit
from functools import partial

print('>>> JAX 運算設備:', jax.devices())

# 匯入您的 Simplex DG 模組
from src.core.generators import get_reference_data
from src.core.connectivity import build_connectivity
from src.bases.vandermonde import vandermonde_2d_dubiner, grad_vandermonde_2d_dubiner
from src.reconstruction import build_differentiation_matrices, build_fmask_table1

Obtaining file:///Users/user/Downloads/%E5%B0%88%E9%A1%8C%E7%9F%AD%E8%AC%9B/0717/%E9%95%B7%E6%99%82%E9%96%93%E6%A8%A1%E6%93%AC/Simplex-DG-solver
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for simplex-dg-solver (pyproject.toml): started
  Building editable for simplex-dg-solver (pyproject.toml): finished with status 'done'
  Created wheel for simplex-dg-solver: filename=simplex_dg_solver-0.2.0-0.editable-py3-none-any.whl size=2944 sha256=f76fed55c788e0b91c4bed1bd1a1a60aaac17809274c6b36b2ea91d855606a9a
  Stored in directory


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Platform 'mps' is experimental and not all JAX functionality may be correctly supported!
[jax-mps] Tip: set JAX_MPS_ASYNC_DISPATCH=1 to opt into async dispatch (experimental; can be much faster for dispatch-bound workloads).


>>> JAX 運算設備: [MpsDevice(id=0)]


## 1. 幾何與球面網格生成 (Spherical Octahedron Simplex Mesh)
生成八面體細分的球面三角形網格，並精確計算 Jacobian 與逆變/協變基向量。

In [2]:
# STREAMING_CHUNK:Defining mesh generation and mappings...
def generate_spherical_octahedron_mesh(n_div: int, R_sphere: float = 1.0):
    v = np.array([
        [1, 0, 0], [-1, 0, 0],
        [0, 1, 0], [0, -1, 0],
        [0, 0, 1], [0, 0, -1]
    ], dtype=float)
    base_faces = [[0, 2, 4], [2, 1, 4], [1, 3, 4], [3, 0, 4], [2, 0, 5], [1, 2, 5], [3, 1, 5], [0, 3, 5]]
    nodes, EToV, node_map = [], [], {}

    def get_node_id(pt):
        pt = np.array(pt)
        pt = R_sphere * pt / np.linalg.norm(pt)
        key = (round(pt[0], 8), round(pt[1], 8), round(pt[2], 8))
        if key not in node_map:
            node_map[key] = len(nodes)
            nodes.append(pt.tolist())
        return node_map[key]

    for face in base_faces:
        v0, v1, v2 = v[face[0]], v[face[1]], v[face[2]]
        for i in range(n_div):
            for j in range(n_div - i):
                p1 = v0 + (v1 - v0) * (i / n_div) + (v2 - v0) * (j / n_div)
                p2 = v0 + (v1 - v0) * ((i + 1) / n_div) + (v2 - v0) * (j / n_div)
                p3 = v0 + (v1 - v0) * (i / n_div) + (v2 - v0) * ((j + 1) / n_div)
                EToV.append([get_node_id(p1), get_node_id(p2), get_node_id(p3)])
                if i > 0:
                    p4 = v0 + (v1 - v0) * (i / n_div) + (v2 - v0) * (j / n_div)
                    p5 = v0 + (v1 - v0) * ((i - 1) / n_div) + (v2 - v0) * ((j + 1) / n_div)
                    p6 = v0 + (v1 - v0) * (i / n_div) + (v2 - v0) * ((j + 1) / n_div)
                    EToV.append([get_node_id(p4), get_node_id(p6), get_node_id(p5)])
    return np.array(nodes), np.array(EToV)

def build_global_index_maps(EToV, EToE, EToF, xi_ref, eta_ref, Np, weights_1d):
    K, nfp = int(EToV.shape[0]), int(len(weights_1d))
    bary_coords = np.column_stack([(-xi_ref - eta_ref) / 2.0, (xi_ref + 1.0) / 2.0, (eta_ref + 1.0) / 2.0])
    fmask = build_fmask_table1(bary_coords)
    vmapM, vmapP = np.zeros((3 * nfp, K), dtype=int), np.zeros((3 * nfp, K), dtype=int)
    for k_elem in range(K):
        for face in range(3):
            interior_indices = k_elem * Np + fmask[:, face]
            vmapM[face * nfp:(face + 1) * nfp, k_elem] = interior_indices
            k_neighbor, f_neighbor = int(EToE[k_elem, face]), int(EToF[k_elem, face])
            if k_neighbor == k_elem:
                vmapP[face * nfp:(face + 1) * nfp, k_elem] = interior_indices
            else:
                neighbor_indices = k_neighbor * Np + fmask[:, f_neighbor]
                vmapP[face * nfp:(face + 1) * nfp, k_elem] = neighbor_indices[::-1]
    return vmapM, vmapP, vmapM == vmapP, fmask

# STREAMING_CHUNK:Defining precise metrics calculation...
def compute_global_coordinates(nodes, EToV, xi_ref, eta_ref, R_sphere):
    K, Np = EToV.shape[0], len(xi_ref)
    X, Y, Z = np.zeros((Np, K)), np.zeros((Np, K)), np.zeros((Np, K))
    L1, L2, L3 = -(xi_ref + eta_ref) / 2.0, (xi_ref + 1.0) / 2.0, (eta_ref + 1.0) / 2.0
    for k in range(K):
        v1, v2, v3 = nodes[EToV[k]]
        x_k = L1 * v1[0] + L2 * v2[0] + L3 * v3[0]
        y_k = L1 * v1[1] + L2 * v2[1] + L3 * v3[1]
        z_k = L1 * v1[2] + L2 * v2[2] + L3 * v3[2]
        r = np.sqrt(x_k**2 + y_k**2 + z_k**2)
        X[:, k], Y[:, k], Z[:, k] = R_sphere * (x_k / r), R_sphere * (y_k / r), R_sphere * (z_k / r)
    return X, Y, Z

def compute_exact_metrics_3d(nodes, EToV, xi_ref, eta_ref, R_sphere=1.0):
    K, Np = EToV.shape[0], len(xi_ref)
    J_exact = np.zeros((Np, K))
    a1x, a1y, a1z, a2x, a2y, a2z = np.zeros((Np, K)), np.zeros((Np, K)), np.zeros((Np, K)), np.zeros((Np, K)), np.zeros((Np, K)), np.zeros((Np, K))
    L1, L2, L3 = -(xi_ref + eta_ref) / 2.0, (xi_ref + 1.0) / 2.0, (eta_ref + 1.0) / 2.0
    for k in range(K):
        v1, v2, v3 = nodes[EToV[k]]
        x_flat, y_flat, z_flat = L1 * v1[0] + L2 * v2[0] + L3 * v3[0], L1 * v1[1] + L2 * v2[1] + L3 * v3[1], L1 * v1[2] + L2 * v2[2] + L3 * v3[2]
        norm_x = np.sqrt(x_flat**2 + y_flat**2 + z_flat**2)
        dr_x, dr_y, dr_z = -0.5 * v1[0] + 0.5 * v2[0], -0.5 * v1[1] + 0.5 * v2[1], -0.5 * v1[2] + 0.5 * v2[2]
        ds_x, ds_y, ds_z = -0.5 * v1[0] + 0.5 * v3[0], -0.5 * v1[1] + 0.5 * v3[1], -0.5 * v1[2] + 0.5 * v3[2]
        cross_x, cross_y, cross_z = dr_y * ds_z - dr_z * ds_y, dr_z * ds_x - dr_x * ds_z, dr_x * ds_y - dr_y * ds_x
        h = v1[0] * cross_x + v1[1] * cross_y + v1[2] * cross_z
        J_exact[:, k] = h / (norm_x**3)
        factor1, factor2 = norm_x / h, norm_x / h
        a1x[:, k], a1y[:, k], a1z[:, k] = factor1 * (ds_y * z_flat - ds_z * y_flat), factor1 * (ds_z * x_flat - ds_x * z_flat), factor1 * (ds_x * y_flat - ds_y * x_flat)
        a2x[:, k], a2y[:, k], a2z[:, k] = factor2 * (y_flat * dr_z - z_flat * dr_y), factor2 * (z_flat * dr_x - x_flat * dr_z), factor2 * (x_flat * dr_y - y_flat * dr_x)
    return J_exact, a1x, a1y, a1z, a2x, a2y, a2z

def compute_h_min(nodes, EToV):
    vertices = nodes[EToV]
    e01 = np.linalg.norm(vertices[:, 1, :] - vertices[:, 0, :], axis=1)
    e12 = np.linalg.norm(vertices[:, 2, :] - vertices[:, 1, :], axis=1)
    e20 = np.linalg.norm(vertices[:, 0, :] - vertices[:, 2, :], axis=1)
    return np.minimum(np.minimum(e01, e12), e20)

## 2. Williamson-1 固體旋轉測試案例與解析解
定義平流速度場與初始高斯鐘形函數 (Gaussian Bell)。

In [3]:
# STREAMING_CHUNK:Defining velocity and initial conditions...
def velocity_solid_body_3d(X, Y, Z, u0, alpha0):
    Omega_x, Omega_y, Omega_z = -np.sin(alpha0) * u0, 0.0, np.cos(alpha0) * u0
    return Omega_y * Z - Omega_z * Y, Omega_z * X - Omega_x * Z, Omega_x * Y - Omega_y * X

@jit
def exact_gaussian_bell_3d_jax(X, Y, Z, t, u0, R, alpha0=0.0):
    Omega = u0 / R
    theta_rot = -Omega * t
    Kx, Ky, Kz = -jnp.sin(alpha0), 0.0, jnp.cos(alpha0)
    dot_KV = Kx * X + Ky * Y + Kz * Z
    KxV_x, KxV_y, KxV_z = Ky * Z - Kz * Y, Kz * X - Kx * Z, Kx * Y - Ky * X
    X_rot = X * jnp.cos(theta_rot) + KxV_x * jnp.sin(theta_rot) + Kx * dot_KV * (1 - jnp.cos(theta_rot))
    Y_rot = Y * jnp.cos(theta_rot) + KxV_y * jnp.sin(theta_rot) + Ky * dot_KV * (1 - jnp.cos(theta_rot))
    Z_rot = Z * jnp.cos(theta_rot) + KxV_z * jnp.sin(theta_rot) + Kz * dot_KV * (1 - jnp.cos(theta_rot))
    dot_product = ((X_rot/R) * 0.0 + (Y_rot/R) * 1.0 + (Z_rot/R) * 0.0)
    dist = R * jnp.arccos(jnp.clip(dot_product, -1.0, 1.0))
    return jnp.exp(-10.0 * (dist / R)**2)

## 3. JAX 核心求解器 (嚴格對應 SBP 數學本質)
利用 JAX 的 JIT 編譯與靜態參數宣告 (static_argnames)，確保 5 階 RK 運算的高效性。

In [4]:
# STREAMING_CHUNK:Building DG RHS and Time Integration Loops...
@partial(jit, static_argnames=["flux_type", "formulation"])
def compute_rhs_3d_jax_optimized(
    q, D_r_ref, D_s_ref, E, J, vmapM, vmapP, weights_1d, M_inv_mat,
    c_r, c_s, vn_sJ, J_M, flux_type="upwind", formulation="divergence", alpha_lf=1.0, global_V_max=1.0
):
    if formulation == "divergence":
        div_r = D_r_ref @ (c_r * q)
        div_s = D_s_ref @ (c_s * q)
        volume_term = -(div_r + div_s) / J
    elif formulation == "split3":
        div_r = D_r_ref @ (c_r * q)
        div_s = D_s_ref @ (c_s * q)
        adv_r = c_r * (D_r_ref @ q)
        adv_s = c_s * (D_s_ref @ q)
        cor_r = D_r_ref @ c_r
        cor_s = D_s_ref @ c_s
        cor_term = q * (cor_r + cor_s)
        volume_term = -0.5 * (div_r + div_s + adv_r + adv_s + cor_term) / J
    elif formulation == "split2":
        div_r = D_r_ref @ (c_r * q)
        div_s = D_s_ref @ (c_s * q)
        adv_r = c_r * (D_r_ref @ q)
        adv_s = c_s * (D_s_ref @ q)
        volume_term = -0.5 * (div_r + div_s + adv_r + adv_s) / J
    else:
        volume_term = jnp.zeros_like(q)

    q_flat = q.T.flatten()
    q_M, q_P = q_flat[vmapM], q_flat[vmapP]

    if flux_type == "upwind":
        C_val_sJ = alpha_lf * jnp.abs(vn_sJ)
    elif flux_type == "LF":
        C_val_sJ = alpha_lf * global_V_max * J_M
    elif flux_type == "central":
        C_val_sJ = 0.0
    else:
        C_val_sJ = 0.0

    penalty_factor_sJ = 0.5 * (vn_sJ - C_val_sJ)
    flux_penalty_sJ = penalty_factor_sJ * (q_M - q_P)

    face_w = jnp.tile(weights_1d, 3)
    scaled_penalty = flux_penalty_sJ * face_w[:, jnp.newaxis]

    surface_integral = E.T @ scaled_penalty
    surface_term = (1.0 / J) * (M_inv_mat @ surface_integral)

    return jnp.expand_dims(volume_term + surface_term, axis=0)

@partial(jit, static_argnames=["flux_type", "formulation", "total_sample_steps", "sub_steps_per_sample"])
def run_full_simulation_scan_optimized(
    Q_init, total_sample_steps, sub_steps_per_sample, dt_rk,
    A_RK, B_RK, D_r_ref, D_s_ref, E, J, vmapM, vmapP, weights_1d, M_inv_mat,
    c_r, c_s, vn_sJ, J_M, flux_type, formulation, global_V_max,
    X, Y, Z, u0, R_sphere, alpha0, M_diag, mass_initial, energy_initial
):
    def inner_rk_step(carry, _):
        Q, t_curr, cum_diss = carry
        du = jnp.zeros_like(Q)
        d_diss = 0.0
        Q_new, cum_diss_new = Q, cum_diss
        face_w = jnp.tile(weights_1d, 3)

        for stage in range(5):
            R_Q = compute_rhs_3d_jax_optimized(
                Q_new[0], D_r_ref, D_s_ref, E, J, vmapM, vmapP, weights_1d, M_inv_mat,
                c_r, c_s, vn_sJ, J_M, flux_type=flux_type, formulation=formulation, alpha_lf=1.0, global_V_max=global_V_max
            )
            q_flat = Q_new[0].T.flatten()
            q_M, q_P = q_flat[vmapM], q_flat[vmapP]
            C_val_diss = jnp.abs(vn_sJ) if flux_type == "upwind" else 0.0
            
            diss_rate = -0.25 * jnp.sum(face_w[:, jnp.newaxis] * C_val_diss * (q_M - q_P)**2)
            
            du = A_RK[stage] * du + dt_rk * R_Q
            d_diss = A_RK[stage] * d_diss + dt_rk * diss_rate
            Q_new = Q_new + B_RK[stage] * du
            cum_diss_new = cum_diss_new + B_RK[stage] * d_diss

        return (Q_new, t_curr + dt_rk, cum_diss_new), None

    def outer_sample_step(carry, _):
        Q_curr, t_curr, cum_diss = carry
        (Q_next, t_next, cum_diss_next), _ = jax.lax.scan(
            inner_rk_step, (Q_curr, t_curr, cum_diss), None, length=sub_steps_per_sample
        )
        Q_exact = exact_gaussian_bell_3d_jax(X / R_sphere, Y / R_sphere, Z / R_sphere, t_next, u0, 1.0, alpha0)
        error_field = Q_next[0] - Q_exact
        
        l2_error = jnp.sqrt(jnp.sum(error_field**2 * M_diag))
        linf_error = jnp.max(jnp.abs(error_field))
        mass_error = (jnp.sum(Q_next[0] * M_diag) - mass_initial) / jnp.abs(mass_initial)
        energy_curr = 0.5 * jnp.sum((Q_next[0]**2) * M_diag)
        energy_error = (energy_curr - energy_initial) / jnp.abs(energy_initial)
        rel_energy_residual = ((energy_curr - energy_initial) - cum_diss_next) / jnp.abs(energy_initial)

        metrics = jnp.array([t_next, l2_error, linf_error, mass_error, energy_error, rel_energy_residual])
        return (Q_next, t_next, cum_diss_next), metrics

    final_state, history = jax.lax.scan(outer_sample_step, (Q_init, 0.0, 0.0), None, length=total_sample_steps)
    return final_state, history

## 4. 主程式整合與正交實驗工作流
設置 5 階段 RK 係數，執行 JAX 加速實驗。

In [ ]:
# STREAMING_CHUNK:Setting up simulation wrapper and LSRK54 coefficients...
# 5-Stage Runge-Kutta 係數
A_RK_np = np.array([0.0, -567301805773.0 / 1357537059087.0, -2404267990393.0 / 2016746695238.0,
                    -3550918686646.0 / 2091501179385.0, -1275806237668.0 / 842570457699.0])
B_RK_np = np.array([1432997174477.0 / 9575080441755.0, 5161836677717.0 / 13612068292357.0,
                    1720146321549.0 / 2090206949498.0, 3134564353537.0 / 4481467310338.0,
                    2277821191437.0 / 14882151754819.0])

def run_manifold_simulation(k_degree=4, n_div=2, t_final=1.0, CFL=0.5, formulation="divergence", flux_type="upwind"):
    R_sphere = 6.37122e6
    nodes, EToV = generate_spherical_octahedron_mesh(n_div, R_sphere=1.0)
    K = EToV.shape[0]

    EToE, EToF = build_connectivity(EToV)
    ref_data = get_reference_data("table1", k_degree)
    xi_ref, eta_ref = ref_data["xi"], ref_data["eta"]
    weights_ref, weights_1d = ref_data["weights"], ref_data["weights_1d"]
    Np, nfp = len(xi_ref), len(weights_1d)

    vmapM, vmapP, _, fmask = build_global_index_maps(EToV, EToE, EToF, xi_ref, eta_ref, Np, weights_1d)
    V_nodal_raw = vandermonde_2d_dubiner(xi_ref, eta_ref, k_degree)
    Vr_raw, Vs_raw = grad_vandermonde_2d_dubiner(xi_ref, eta_ref, k_degree)
    M_modal_approx = V_nodal_raw.T @ np.diag(weights_ref) @ V_nodal_raw
    inv_L_T = np.linalg.inv(np.linalg.cholesky(M_modal_approx).T)
    V_nodal, Vr, Vs = V_nodal_raw @ inv_L_T, Vr_raw @ inv_L_T, Vs_raw @ inv_L_T

    E = np.zeros((3 * nfp, Np))
    for face in range(3):
        for local_i, node_idx in enumerate(fmask[:, face]):
            E[face * nfp + local_i, node_idx] = 1.0

    nr_expanded = np.repeat(np.array([0.0, 1.0, -1.0]), nfp)[:, np.newaxis]
    ns_expanded = np.repeat(np.array([-1.0, 1.0, 0.0]), nfp)[:, np.newaxis]
    M_diag_mat = np.diag(weights_ref)
    M_inv_mat = np.diag(1.0 / weights_ref)
    VVT = V_nodal @ V_nodal.T
    face_weights_expanded = np.tile(weights_1d, 3)
    B1 = E.T @ np.diag(face_weights_expanded * nr_expanded.flatten()) @ E
    B2 = E.T @ np.diag(face_weights_expanded * ns_expanded.flatten()) @ E
    term_pre = 0.5 * (M_inv_mat + VVT)
    term_post = (np.eye(Np) - VVT @ M_diag_mat)
    
    D_r_ref = term_pre @ B1 @ term_post + Vr @ V_nodal.T @ M_diag_mat
    D_s_ref = term_pre @ B2 @ term_post + Vs @ V_nodal.T @ M_diag_mat

    X, Y, Z = compute_global_coordinates(nodes, EToV, xi_ref, eta_ref, R_sphere)
    J, a1x, a1y, a1z, a2x, a2y, a2z = compute_exact_metrics_3d(nodes, EToV, xi_ref, eta_ref, R_sphere=1.0)

    u0 = (2 * np.pi * 1.0) / (12 * 86400)
    alpha0 = np.pi / 4.0
    U_3d, V_3d, W_3d = velocity_solid_body_3d(X/R_sphere, Y/R_sphere, Z/R_sphere, u0, alpha0)
    V_max = float(np.max(np.sqrt(U_3d**2 + V_3d**2 + W_3d**2)))
    
    dt_global = CFL * float(np.min(compute_h_min(nodes, EToV))) / (V_max * (k_degree + 1)**2)

    tilde_u = a1x * U_3d + a1y * V_3d + a1z * W_3d
    tilde_v = a2x * U_3d + a2y * V_3d + a2z * W_3d
    c_r_np, c_s_np = J * tilde_u, J * tilde_v
    J_M_np = J.T.flatten()[vmapM]
    vn_sJ_np = J_M_np * (nr_expanded * tilde_u.T.flatten()[vmapM] + ns_expanded * tilde_v.T.flatten()[vmapM])
    # vn_sJ_np_p = J_M_np * (nr_expanded * tilde_u.T.flatten()[vmapP] + ns_expanded * tilde_v.T.flatten()[vmapP])
    # print(f"Max error: {np.max(np.abs(vn_sJ_np - vn_sJ_np_p))}")

    Q_init = exact_gaussian_bell_3d_jax(X, Y, Z, 0.0, u0, R_sphere, alpha0)
    Q_j = jnp.array(Q_init[np.newaxis, :, :])
    M_diag = weights_ref[:, np.newaxis] * J
    M_diag_j = jnp.array(M_diag)

    mass_initial = float(np.sum(Q_init * M_diag))
    energy_initial = float(np.sum(0.5 * (Q_init**2) * M_diag))

    dt_sample = 7200.0 # 每 2 小時紀錄一次
    total_sample_steps = int(np.ceil(t_final / dt_sample))
    actual_dt_sample = t_final / total_sample_steps
    sub_steps_per_sample = int(np.ceil(actual_dt_sample / dt_global))
    actual_dt_rk = actual_dt_sample / sub_steps_per_sample

    final_state, history_j = run_full_simulation_scan_optimized(
        Q_j, total_sample_steps, sub_steps_per_sample, actual_dt_rk,
        jnp.array(A_RK_np), jnp.array(B_RK_np), jnp.array(D_r_ref), jnp.array(D_s_ref), jnp.array(E), jnp.array(J),
        jnp.array(vmapM), jnp.array(vmapP), jnp.array(weights_1d), jnp.array(M_inv_mat),
        jnp.array(c_r_np), jnp.array(c_s_np), jnp.array(vn_sJ_np), jnp.array(J_M_np),
        flux_type, formulation, V_max,
        jnp.array(X), jnp.array(Y), jnp.array(Z), float(u0), float(R_sphere), float(alpha0), M_diag_j, mass_initial, energy_initial
    )

    jax.block_until_ready(history_j)
    history_np = np.asarray(history_j)

    return {
        "time": history_np[:, 0], "l2": history_np[:, 1], "linf": history_np[:, 2],
        "mass_err": history_np[:, 3], "energy_err": history_np[:, 4], "energy_res": history_np[:, 5]
    }


## 5. 啟動 120 天長週期 6-Case 正交實驗與資料匯出

In [6]:
# ==================== 啟動大規模正交實驗 ====================
n_div_list = [64]
t_final = 120.0 * 86400.0  # 120 Days

cases = [
    {'form': 'divergence', 'flux': 'upwind',  'label': 'Cons_Upwind'},
    {'form': 'divergence', 'flux': 'central', 'label': 'Cons_Central'},
    {'form': 'split2',     'flux': 'upwind',  'label': 'Split2_Upwind'},
    {'form': 'split2',     'flux': 'central', 'label': 'Split2_Central'},
    {'form': 'split3',     'flux': 'upwind',  'label': 'Split3_Upwind'},
    {'form': 'split3',     'flux': 'central', 'label': 'Split3_Central'}
]

for n_div in n_div_list:
    print(f'\n================ 網格密度 N = {n_div} ================')
    grid_results = {}
    filename = f'simulation_results_N{n_div}.npz'
    
    for c in cases:
        label = c['label']
        t0 = time.time()
        
        # 強制使用 Keyword Arguments，避免位置參數錯位導致通量計算錯誤
        res = run_manifold_simulation(
            k_degree=4, 
            n_div=n_div, 
            t_final=t_final, 
            CFL=0.5, 
            formulation=c['form'], 
            flux_type=c['flux']
        )
        
        grid_results[f'{label}_time'] = res['time']
        grid_results[f'{label}_l2'] = res['l2']
        grid_results[f'{label}_linf'] = res['linf']
        grid_results[f'{label}_mass_err'] = res['mass_err']
        grid_results[f'{label}_energy_err'] = res['energy_err']
        grid_results[f'{label}_energy_res'] = res['energy_res']
        print(f"  [Case {label}] 耗時: {time.time()-t0:.2f} s | L2 終端誤差: {res['l2'][-1]:.4e}")
        
        # 每次跑完單一 Case，立刻覆寫更新壓縮檔 (進度防丟失)
        np.savez_compressed(filename, **grid_results)
        print(f'   >>> 已將 {label} 數據即時寫入: {filename}')
        
        # 每次跑完單一 Case，立刻清空 JAX 佔用的 VRAM 記憶體
        del res
        gc.collect()
        try: 
            jax.clear_caches()
        except AttributeError: 
            pass
            
    # 完成該 n_div 的所有 Case 後，清空 Host 端的字典記憶體
    del grid_results

print('\n>>> [完成] 大規模實驗跑測與硬碟落盤已全部結束。')


================ 網格密度 N = 64 ================
Max error: 1.1593239385777641e-07


JaxRuntimeError: INTERNAL: BufferFromHostBuffer failed: MLX does not support float64 (F64). Use jax.config.update('jax_enable_x64', False) or ensure your arrays are float32.